In [ ]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import keras_tuner as kt
import tensorboard

In [6]:
df_train = pd.read_csv(r'../data/selected_col/model_train.csv')
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')
df_val = pd.read_csv(r'../data/selected_col/model_val.csv')

In [7]:
img_size = (224,224)
batch_size = 32

In [8]:
batch_size_64 = 64

In [9]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(factor=0.15, value_range=(0.0, 1.0)),
])

In [10]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [11]:
def preprocess_train(image_path, label):
    image, label = preprocess_image(image_path, label)
    image = data_augmentation(image)
    return image, label

In [12]:
def preprocess_test(image_path, label):
    image, label = preprocess_image(image_path, label)
    return image, label

In [13]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset = (
    train_dataset
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [14]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset = (
    test_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [15]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset = (
    val_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [16]:
#batch size 64

In [17]:
batch_size_64 = 64

In [18]:
train_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset_64 = (
    train_dataset_64
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [19]:
test_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset_64 = (
    test_dataset_64
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [20]:
val_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset_64 = (
    val_dataset_64
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [21]:
image,label = next(iter(test_dataset_64))

print(image.shape)

(64, 224, 224, 3)


In [22]:
image,label = next(iter(test_dataset))

print(image.shape)

(32, 224, 224, 3)


In [23]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}

In [24]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [25]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [26]:
#data pipeline 2

In [27]:
data_augmentation_2 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
])

In [28]:
def preprocess_image2(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    return image, label

In [29]:
def preprocess_train2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    image = data_augmentation_2(image)
    return image, label

In [30]:
def preprocess_test2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    return image, label

In [31]:
train_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2 = (
    train_dataset2
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [32]:
test_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2 = (
    test_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [33]:
val_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2 = (
    val_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [34]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model_early = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model_early.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [35]:
mobilenet_model_early.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [36]:
history = mobilenet_model_early.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 225s 984ms/step - accuracy: 0.3207 - loss: 1.7969 - val_accuracy: 0.5110 - val_loss: 1.3958
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 823ms/step - accuracy: 0.4200 - loss: 1.5156 - val_accuracy: 0.5642 - val_loss: 1.2635
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 180s 807ms/step - accuracy: 0.4624 - loss: 1.4275 - val_accuracy: 0.4385 - val_loss: 1.3501
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 799ms/step - accuracy: 0.4573 - loss: 1.3411 - val_accuracy: 0.5795 - val_loss: 1.1223
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 814ms/step - accuracy: 0.4527 - loss: 1.3038 - val_accuracy: 0.5782 - val_loss: 1.1497
Restoring model weights from the end of the best epoch: 4.


In [37]:
mobilenet_model_early.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 32s 686ms/step - accuracy: 0.5722 - loss: 1.1204


[1.1203925609588623, 0.5721889734268188]

In [38]:
mobilenet_model_early.save(r'../models/mobile_net_early.keras')

In [39]:
history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/mibile_net_early.csv',index=False)


In [ ]:
'''lr'''

In [40]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model_learning = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model_learning.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [41]:
mobilenet_model_learning.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [42]:
history = mobilenet_model_learning.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict,callbacks=[lr_scheduler])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 823ms/step - accuracy: 0.3575 - loss: 1.7437 - val_accuracy: 0.4604 - val_loss: 1.4499 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 797ms/step - accuracy: 0.4588 - loss: 1.4763 - val_accuracy: 0.2854 - val_loss: 1.4366 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 806ms/step - accuracy: 0.4584 - loss: 1.4195 - val_accuracy: 0.5642 - val_loss: 1.2124 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 138s 622ms/step - accuracy: 0.4563 - loss: 1.3872 - val_accuracy: 0.5356 - val_loss: 1.1731 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 610ms/step - accuracy: 0.4697 - loss: 1.3297 - val_accuracy: 0.6181 - val_loss: 1.0820 - learning_rate: 0.0010


In [43]:
mobilenet_model_learning.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 30s 636ms/step - accuracy: 0.6081 - loss: 1.0754


[1.0754263401031494, 0.6081171035766602]

In [45]:
mobilenet_model_learning.save(r'../models/mobile_learning.keras')

In [46]:
history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/mibile_net_learning.csv',index=False)


In [ ]:
'''adam'''

In [47]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model_adam = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model_adam.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [48]:
mobilenet_model_adam.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [49]:
history = mobilenet_model_adam.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 800ms/step - accuracy: 0.3166 - loss: 1.7853 - val_accuracy: 0.5050 - val_loss: 1.4246
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 816ms/step - accuracy: 0.4337 - loss: 1.4912 - val_accuracy: 0.4844 - val_loss: 1.2887
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 823ms/step - accuracy: 0.4343 - loss: 1.4278 - val_accuracy: 0.4637 - val_loss: 1.3237
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 817ms/step - accuracy: 0.4653 - loss: 1.3865 - val_accuracy: 0.4019 - val_loss: 1.3817
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 178s 799ms/step - accuracy: 0.4690 - loss: 1.3515 - val_accuracy: 0.5163 - val_loss: 1.1914


In [50]:
mobilenet_model_adam.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 30s 632ms/step - accuracy: 0.5183 - loss: 1.1768


[1.1767609119415283, 0.5182967185974121]

In [51]:
mobilenet_model_adam.save(r'../models/mobile_net_adam.keras')

In [ ]:
'sgd'

In [52]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model_sgd = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model_sgd.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [53]:
mobilenet_model_sgd.compile(optimizer=tf.keras.optimizers.SGD(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [55]:
history = mobilenet_model_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 829ms/step - accuracy: 0.2965 - loss: 1.8028 - val_accuracy: 0.5882 - val_loss: 1.3750
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 821ms/step - accuracy: 0.4253 - loss: 1.5525 - val_accuracy: 0.5689 - val_loss: 1.3440
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 181s 814ms/step - accuracy: 0.4403 - loss: 1.4320 - val_accuracy: 0.6480 - val_loss: 1.0421
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 818ms/step - accuracy: 0.4752 - loss: 1.3556 - val_accuracy: 0.2861 - val_loss: 1.7800
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 182s 819ms/step - accuracy: 0.4737 - loss: 1.3211 - val_accuracy: 0.5030 - val_loss: 1.2576


In [56]:
mobilenet_model_sgd.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 30s 633ms/step - accuracy: 0.5083 - loss: 1.2419


[1.2418842315673828, 0.508316695690155]

In [57]:
mobilenet_model_sgd.save(r'../models/mobile_net_sgd.keras')

In [ ]:
'''rms prop'''

In [58]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model_rms = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model_rms.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [59]:
mobilenet_model_rms.compile(optimizer=tf.keras.optimizers.RMSprop(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [60]:
history = mobilenet_model_rms.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 814ms/step - accuracy: 0.4223 - loss: 1.7863 - val_accuracy: 0.6600 - val_loss: 1.0038
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 832ms/step - accuracy: 0.4771 - loss: 1.5693 - val_accuracy: 0.6780 - val_loss: 0.9777
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 826ms/step - accuracy: 0.5186 - loss: 1.4456 - val_accuracy: 0.6407 - val_loss: 0.9849
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 827ms/step - accuracy: 0.5158 - loss: 1.4550 - val_accuracy: 0.4890 - val_loss: 1.2449
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 828ms/step - accuracy: 0.5172 - loss: 1.4022 - val_accuracy: 0.6760 - val_loss: 0.9363


In [61]:
mobilenet_model_rms.evaluate(test_dataset)

47/47 ━━━━━━━━━━━━━━━━━━━━ 30s 648ms/step - accuracy: 0.6660 - loss: 0.9216


[0.9215537309646606, 0.6660013198852539]

In [62]:
mobilenet_model_rms.save(r'../models/mobile_net_rms.keras')

In [63]:
history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/mibile_net_rms.csv',index=False)


In [ ]:
'''64'''

In [67]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

mobilenet_model_64 = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

mobilenet_model_64.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [68]:
mobilenet_model_64.compile(optimizer=tf.keras.optimizers.RMSprop(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [69]:
history = mobilenet_model_64.fit(train_dataset_64,validation_data=val_dataset_64,epochs=5,class_weight=class_weight_dict)

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.3855 - loss: 1.8163 - val_accuracy: 0.4558 - val_loss: 1.3476
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.4768 - loss: 1.5245 - val_accuracy: 0.6367 - val_loss: 1.0299
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.4934 - loss: 1.4404 - val_accuracy: 0.5436 - val_loss: 1.1573
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 195s 2s/step - accuracy: 0.5272 - loss: 1.3414 - val_accuracy: 0.5323 - val_loss: 1.1801
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.5116 - loss: 1.3278 - val_accuracy: 0.6248 - val_loss: 1.0053


In [70]:
mobilenet_model_64.evaluate(test_dataset_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.6174 - loss: 1.0029


[1.0029265880584717, 0.6174318194389343]

In [71]:
mobilenet_model_64.save(r'../models/mobile_net64.keras')

In [ ]:
'''hyper parameter'''

In [79]:
INPUT_SHAPE = (224, 224, 3)
NUM_CLASSES = 7

def build_mobilenet(hp):

    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    base_model.trainable = False

    model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            units= hp.Choice(
                "dense_units",
                values = [128,256,512]
            ),
            activation="relu"
        ),

        layers.Dropout(
            hp.Choice(
                "dropout",
               values=[0.2,0.3,0.5]
            )
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])

    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        values=["adam", "rmsprop", "sgd"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [80]:
tuner = kt.RandomSearch(
    build_mobilenet,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="mobilenetv2"
)

In [81]:
tuner.search(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict
)

Trial 5 Complete [00h 19m 14s]
val_accuracy: 0.6793080568313599

Best val_accuracy So Far: 0.6793080568313599
Total elapsed time: 01h 36m 28s


In [82]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print(best_hps.values)

{'dense_units': 128, 'dropout': 0.2, 'optimizer': 'rmsprop'}


In [83]:
best_model = tuner.get_best_models(1)[0]

c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [88]:
history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 227s 1s/step - accuracy: 0.6001 - loss: 1.0747 - val_accuracy: 0.7239 - val_loss: 0.8534
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - accuracy: 0.5895 - loss: 1.0690 - val_accuracy: 0.6680 - val_loss: 0.9352
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.5982 - loss: 1.0518 - val_accuracy: 0.6420 - val_loss: 0.9716
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.5955 - loss: 1.0395 - val_accuracy: 0.6733 - val_loss: 0.9074
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [85]:
test_loss, test_accuracy = best_model.evaluate(test_dataset)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 819ms/step - accuracy: 0.5981 - loss: 1.0884
Test Accuracy : 0.5981370806694031
Test Loss : 1.0884037017822266
